# Evaluating LLM Agents: From Vibes to Metrics

**Section 2**: Evals & Evaluation Frameworks

---

> This notebook covers the **theory and concepts** behind evaluating LLM agents.  
> All practical eval code lives in the `evals/` test files and will be run live via `pytest`.

---

## Our Project: Cortex

We're building **Cortex**, a general-purpose multi-agent AI assistant on top of LangGraph. It handles a broad range of user requests — small talk, factual lookups, math and coding problems, and questions about LLM prompt caching — by dispatching each turn to a specialist agent.

```
User Query
    │
    ▼
┌──────────┐
│  Router  │  ← Classifies intent via structured output
└──────────┘
    │
    ├─ general_chat ─────► Generalist     (get_current_time)
    ├─ knowledge_query ──► Researcher     (search_knowledge_base, wikipedia_search)
    ├─ reasoning_task ───► Reasoner       (calculator)
    └─ prompt_caching ───► PromptCacher   (large stable system prompt)
```

The agents are backed by a **PostgreSQL + pgvector** knowledge base of curated articles (LLMs, RAG, prompt caching, multi-agent systems, etc.). Each agent has a YAML-defined persona, rules, and allowed tools.

### Why does this project need evals?

This system has all the ingredients for silent failures:

- **Routing decisions** — if the router misclassifies intent, the wrong specialist handles the request and answer quality collapses
- **Tool sequencing** — the Researcher must call `search_knowledge_base` *before* answering; the Reasoner must call `calculator` for non-trivial arithmetic instead of guessing
- **Grounded responses** — the Researcher's answers must come from retrieved articles, not hallucinated facts
- **Reasoning correctness** — multi-step math and logic must produce the right answer, not a plausible-sounding wrong one
- **Security** — user messages and tool outputs can contain prompt-injection payloads that try to override the agent's instructions

None of these failure modes are visible in the chat UI. The agent always responds confidently. **Evals are how we catch what our eyes miss.**


---

## Why Traditional Testing Fails for Agents

Traditional software testing relies on **deterministic behavior**: given input X, expect output Y. LLM agents break this in fundamental ways:

| Traditional Software | LLM Agent |
|---|---|
| Deterministic output | Stochastic — same input, different outputs |
| Binary pass/fail | Subjective quality — \"good enough\" vs \"perfect\" |
| Fixed control flow | Dynamic tool selection — agent *chooses* what to do |
| Input validation | Natural language — infinite input space |
| Unit testable functions | Multi-step reasoning chains |

### Example from our agent

Consider: *\"What is retrieval-augmented generation?\"*

All of these are correct responses:
- \"RAG retrieves relevant documents at query time and grounds the model in them.\"
- \"Retrieval-Augmented Generation pulls context from a vector store and injects it into the prompt before generation.\"
- \"It's a pattern where you embed your data, search at query time, and pass the top results plus the question to the LLM.\"

**None would pass an exact-match test.** We need a fundamentally different approach — one that evaluates *quality*, *grounding*, and *correctness* rather than exact string matching.


---

## The Evaluation Taxonomy

### What to Evaluate in an Agent System

Evaluation for agents happens at multiple levels — from low-level tool calls to end-to-end user experience:

```
┌───────────────────────────────┐
│  Level 4: End-to-End Quality    │  ← Did the user get a useful, correct answer?
├───────────────────────────────┤
│  Level 3: Response Quality      │  ← Is the response faithful, relevant, hallucination-free?
├───────────────────────────────┤
│  Level 2: Tool Usage            │  ← Did the agent call the right tools with the right args?
├───────────────────────────────┤
│  Level 1: Routing               │  ← Did the router pick the correct specialist agent?
└───────────────────────────────┘
```

### When to Evaluate

| Stage | Method | Speed | Coverage |
|-------|--------|-------|----------|
| **Development** | Golden dataset + offline evals | Fast | Fixed scenarios |
| **CI/CD** | Automated test suite (`pytest` / `deepeval test run`) | Minutes | Regression detection |
| **Production** | Online eval (sample + LLM-as-judge) | Async | Real traffic |
| **Review** | Human evaluation + labeling | Slow | Ground truth creation |

### LLM-as-Judge: The Key Concept

Since we can't use exact matching, we use **another LLM to evaluate the agent's output**. This is called **LLM-as-Judge**.

```
User Question ──┐
                 ├──► Judge LLM ──► Score (0.0 - 1.0)
Agent Response ──┤                  + Reasoning
Ground Truth ────┘
```

The judge receives the question, the agent's response, and optionally ground truth data, then produces a score with reasoning. Most evaluation frameworks are built around this pattern.


---

## Types of Eval Metrics

There's no single metric that tells you \"the agent is good.\" Different aspects of an agent require different kinds of evaluation. Here's a breakdown of the major metric categories:

### Agent Metrics (Tool Use & Orchestration)

These measure whether the agent took the **right actions** — not just whether it said the right thing.

| Metric | What It Measures |
|--------|------------------|
| **Tool Correctness** | Did the agent call the right tools in the right order with the right arguments? |
| **Task Completion** | Did the agent actually accomplish the user's goal end-to-end? |
| **Policy Adherence** | Did the agent follow its rules (e.g. \"always cite a source\", \"use the calculator for non-trivial arithmetic\")? |

```
Expected (researcher): search_knowledge_base(\"prompt caching\") → cite source → reply
Actual:                reply directly from memory                                      ← FAIL: skipped retrieval
```

### RAG Metrics (Retrieval-Augmented Generation)

These measure whether the agent's response is **grounded in retrieved data** rather than hallucinated.

| Metric | What It Measures |
|--------|------------------|
| **Faithfulness** | Are all claims in the response supported by the retrieved context? |
| **Hallucination** | Did the agent make up facts not present in any provided context? |
| **Context Precision** | Is the retrieved context actually relevant to the question? |
| **Context Recall** | Does the retrieved context contain all the info needed to answer? |
| **Answer Relevancy** | Does the response actually address what the user asked? |

```
Retrieved:  \"Prompt caching reduces TTFT by reusing KV state for identical prefixes.\"
✓ Faithful: \"Prompt caching speeds up time-to-first-token by reusing the KV cache for identical prefixes.\"
✗ Hallucinated: \"Prompt caching makes the model cheaper because it skips embedding lookup.\" (claim not in context)
```

### Reasoning Metrics

For the Reasoner specialist, correctness of the final numerical or logical answer matters more than prose quality.

| Metric | What It Measures |
|--------|------------------|
| **Answer Correctness** | For arithmetic and logic problems, does the final answer match the ground truth? |
| **Chain-of-Thought Quality** | Are the intermediate steps coherent and consistent with the final answer? |
| **Tool Use** | For multi-digit arithmetic, did the agent call `calculator` instead of guessing? |

### Multi-Turn Metrics

Single-turn evals miss a critical dimension — can the agent maintain context across a conversation?

| Metric | What It Measures |
|--------|------------------|
| **Context Retention** | Does the agent remember earlier parts of the conversation? |
| **Disambiguation** | Can the agent resolve references like \"that paper\" or \"the second example\"? |
| **Conversation Coherence** | Does the agent's behavior stay consistent across turns? |

### Safety & Security Metrics

These measure whether the agent can withstand adversarial inputs and stay within bounds.

| Metric | What It Measures |
|--------|------------------|
| **Prompt Injection Resilience** | Does the agent ignore injected instructions in tool outputs or user messages? |
| **Toxicity** | Does the agent generate harmful, offensive, or biased content? |
| **PII Leakage** | Does the agent expose sensitive information it shouldn't? |
| **Scope Adherence** | Does the agent refuse off-topic or out-of-policy requests appropriately? |

### MCP (Model Context Protocol) Metrics

As agents connect to external tools via MCP, new evaluation dimensions emerge:

| Metric | What It Measures |
|--------|------------------|
| **Tool Selection Accuracy** | Does the agent pick the right MCP tool from a large registry? |
| **Parameter Extraction** | Are tool arguments correctly extracted from natural language? |
| **Error Recovery** | Does the agent handle tool failures or unexpected responses gracefully? |

### Custom / Domain-Specific Metrics (GEval)

Not everything maps to a built-in metric. **GEval** lets you define a custom rubric and use an LLM as judge.

| Metric | What It Measures |
|--------|------------------|
| **Conciseness** | Is the response as short as possible while still complete? |
| **Citation Discipline** | Does the Researcher always end with a Sources line? |
| **Show-Your-Work** | Does the Reasoner present intermediate steps for non-trivial problems? |

```
Evaluation Criteria: Rate the Researcher response on citation discipline (1-5).
- 5: Every factual claim is followed by an inline source or a Sources section
- 3: Some claims sourced, others left dangling
- 1: No sources cited at all
```


---

## Evaluation Frameworks: A Comparison

There are several frameworks available for evaluating LLM applications. Here's how the major ones compare:

| Framework | Best For | Key Strengths | Limitations |
|-----------|----------|---------------|-------------|
| **DeepEval** | Agent evals, CI/CD integration | Tool correctness, GEval custom metrics, pytest-native, built-in dashboard | Smaller community than some alternatives |
| **RAGAS** | RAG pipeline evaluation | Claim-level faithfulness decomposition, comprehensive RAG metrics | Focused primarily on RAG, less on agent orchestration |
| **LangSmith** | LangChain ecosystem | Tight LangChain integration, tracing + evals in one platform | Vendor lock-in to LangChain ecosystem |
| **Promptfoo** | Prompt engineering & red-teaming | Config-driven, great for prompt comparison, security testing | Less focus on agent-level orchestration |
| **Braintrust** | Production monitoring | Logging + evals combined, real-time scoring | Primarily a hosted platform |
| **Arize Phoenix** | Observability + evals | Tracing and eval in one tool, open-source | More observability-focused than eval-focused |

### How to Choose?

Ask yourself these questions:

1. **What are you evaluating?**
   - RAG pipeline → **RAGAS** excels at faithfulness and context quality
   - Agent tool use → **DeepEval** has `ToolCorrectnessMetric` built-in
   - Prompt quality → **Promptfoo** is great for A/B testing prompts

2. **Where do you need evals?**
   - CI/CD pipeline → **DeepEval** (pytest integration, `deepeval test run`)
   - Production monitoring → **Braintrust** or **Arize Phoenix**
   - Development iteration → **LangSmith** or **RAGAS**

3. **How custom are your metrics?**
   - Standard metrics (faithfulness, relevancy) → Most frameworks cover these
   - Custom rubrics (citation discipline, scope adherence) → **DeepEval's GEval** or **RAGAS** custom metrics

### Our Choice: DeepEval (primary)

In this project, we use **DeepEval** as our primary framework because:
- It integrates natively with **pytest** — our tests are just test functions
- It has **`ToolCorrectnessMetric`** — purpose-built for evaluating agent tool calls
- It supports **GEval** — letting us define custom rubrics for routing accuracy, faithfulness, and injection resilience
- It provides a **dashboard** via `deepeval test run` for tracking scores over time

For the **Researcher** agent (our RAG specialist), DeepEval's `FaithfulnessMetric` checks whether claims are grounded in the retrieved knowledge-base articles. RAGAS is an excellent complement when you want claim-level decomposition, but for Cortex we keep the eval surface minimal and DeepEval-only.

| Aspect | DeepEval | RAGAS |
|--------|----------|-------|
| **Integration** | pytest via `assert_test()` | `evaluate()` returns DataFrame |
| **Granularity** | Single score per metric | Claim-level decomposition |
| **Best for** | CI/CD gating, agent evals | Detailed RAG analysis and debugging |
| **In our project** | `test_routing.py`, `test_faithfulness.py`, `test_security.py` | (optional, not used in Cortex) |


---

## DeepEval: A Quick Introduction

[DeepEval](https://github.com/confident-ai/deepeval) is an open-source evaluation framework for LLM applications. It's designed to work like a **unit testing framework** — you write test cases, define metrics, set thresholds, and run them with `pytest`.

### Core Concepts

**1. `LLMTestCase`** — The unit of evaluation

Each test case wraps the input, the agent's output, and any supporting context:

```python
from deepeval.test_case import LLMTestCase

test_case = LLMTestCase(
    input="What is prompt caching?",
    actual_output="Prompt caching reuses KV state for identical prefixes, reducing TTFT.",
    expected_output="Prompt caching speeds up time-to-first-token by reusing the KV cache.",
    retrieval_context=["Prompt caching reduces TTFT by reusing KV state for identical prefixes."],
    expected_tools=["search_knowledge_base"],
    actual_tools=["search_knowledge_base"],
)
```

**2. Metrics** — What to measure

DeepEval provides built-in metrics that use LLM-as-Judge under the hood:

| Metric | Purpose |
|--------|---------|
| `ToolCorrectnessMetric` | Compares expected vs actual tool calls |
| `FaithfulnessMetric` | Checks if response is grounded in retrieval context |
| `HallucinationMetric` | Detects fabricated information |
| `AnswerRelevancyMetric` | Checks if response addresses the question |
| `GEval` | Custom rubric — you define the criteria |

**3. `assert_test()`** — The assertion

```python
from deepeval import assert_test

assert_test(test_case, metrics=[ToolCorrectnessMetric()])
```

If the metric score falls below the threshold, the test fails — just like a regular assertion.

**4. Running evals**

```bash
# Via pytest (what we use)
pytest evals/test_routing.py -v

# Via deepeval CLI (adds dashboard + reporting)
deepeval test run evals/ --report
```

### GEval: Build Your Own Metric

GEval is DeepEval's most flexible feature — it lets you define **any evaluation criterion** as natural language:

```python
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

citation_metric = GEval(
    name="Citation Discipline",
    criteria="Evaluate whether the response cites the knowledge-base article it drew "
             "facts from, and refuses to answer when no relevant context was retrieved.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.RETRIEVAL_CONTEXT],
    threshold=0.7,
)
```

The LLM judge reads your criteria and scores the response accordingly. This is how we evaluate things like **routing accuracy**, **citation discipline**, and **injection resilience** — dimensions that no pre-built metric covers.

---

## Let's See It in Action

Now that we understand the theory — **why** we need evals, **what** types of metrics exist, and **which** frameworks to use — let's run them against our live agent.

### What we'll do in the practical demo

We'll run `pytest` against the running LangGraph server to evaluate Cortex across multiple dimensions:

| Eval | Test File | What It Catches |
|------|-----------|----------------|
| Routing Accuracy | `evals/test_routing.py` | Router misclassifying user intent and dispatching to the wrong specialist |
| Faithfulness | `evals/test_faithfulness.py` | Researcher answering without consulting the knowledge base, or contradicting retrieved context |
| Security | `evals/test_security.py` | Direct prompt injection that tries to override the agent's instructions |

### The pattern for every demo

1. **Chat with the agent** in the UI — it looks fine
2. **Run the eval** — it reveals silent failures
3. **Fix the agent** — update the YAML config or middleware
4. **Re-run the eval** — scores improve

> **The gap between \"looks fine in chat\" and \"passes evals\" is exactly why we need automated evaluation.**
